# NTT (9432.T) デイトレード・バックテスト

オープニングレンジ・ブレイクアウト戦略を5分足で検証します。
**iPad / スマホからそのまま実行できます。**

## 使い方

1. 上のメニューから **「ランタイム」→「すべてのセルを実行」** をタップ
2. 数十秒待つと、成績サマリー・資産曲線・トレード履歴が表示されます

設定を変えたいときは「2. パラメータ設定」のスライダーや入力欄を触ってから、
そのセル以降を実行し直してください。

---

> **免責**: 本ノートブックは戦略検証の教材であり、投資助言ではありません。
> 過去の成績は将来の成果を保証しません。


## 1. セットアップ

必要なライブラリを入れて、GitHubからバックテスト本体を取得します。


In [ ]:
#@title 実行するとセットアップされます { display-mode: "form" }import os, subprocess, sys, textwrapREPO   = "https://github.com/furiya0530-oss/-SNS.git"BRANCH = "main"   #@param {type:"string"}WORKDIR = "/content/ntt-backtest"SCRIPT  = "ntt_daytrade_backtest.py"def sh(cmd):    p = subprocess.run(cmd, capture_output=True, text=True)    return p.returncode, (p.stdout + p.stderr)print("[1/2] yfinance をインストール中 ...")rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "yfinance"])print("      " + ("完了" if rc == 0 else "失敗:\n" + out))print("[2/2] バックテスト本体を取得中 ...")if not os.path.isdir(WORKDIR):    # リポジトリ名が '-' 始まりなので、明示的なディレクトリ名にクローンする    rc, out = sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, WORKDIR])    if rc != 0:        print("      clone 失敗:\n" + out)else:    sh(["git", "-C", WORKDIR, "pull", "--ff-only"])if os.path.isfile(os.path.join(WORKDIR, SCRIPT)):    os.chdir(WORKDIR)    print(f"      完了 -> {WORKDIR}")    print("\n準備ができました。次のセルへ進んでください。")else:    print(textwrap.dedent("""          スクリプトを取得できませんでした。          左のフォルダアイコンから ntt_daytrade_backtest.py を          アップロードするか、次を実行してください:              from google.colab import files; files.upload()          """))

## 2. パラメータ設定

スライダーとメニューで設定できます。変更したら**このセルを実行**してから次へ。


In [ ]:
#@title バックテスト設定 { display-mode: "form", run: "auto" }#@markdown ### 銘柄とデータ期間ticker   = "9432.T"  #@param {type:"string"}period   = "60d"     #@param ["5d", "1mo", "60d"]interval = "5m"      #@param ["1m", "5m", "15m", "30m", "60m"]#@markdown ### 戦略#@markdown 寄り付きから何分間をレンジとするかor_minutes  = 30   #@param {type:"slider", min:5, max:120, step:5}#@markdown 損切り／利確（レンジ幅の倍数）stop_mult   = 1.0  #@param {type:"number"}target_mult = 2.0  #@param {type:"number"}#@markdown ダマシ除けのバッファ (bps)buffer_bps  = 5.0  #@param {type:"number"}allow_short = True #@param {type:"boolean"}#@markdown ### 資金とコストcapital        = 1000000 #@param {type:"integer"}#@markdown 1トレードあたりのリスク (％)risk_pct       = 1.0     #@param {type:"slider", min:0.1, max:5.0, step:0.1}commission_bps = 5.0     #@param {type:"number"}slippage_bps   = 3.0     #@param {type:"number"}# --- Yahoo の期間制限チェック（1分足は7日までなど） ---MAX_DAYS = {"1m": 7, "5m": 60, "15m": 60, "30m": 60, "60m": 730}PERIOD_DAYS = {"5d": 5, "1mo": 30, "60d": 60}if PERIOD_DAYS[period] > MAX_DAYS[interval]:    print(f"[注意] {interval}足は約{MAX_DAYS[interval]}日分までしか取得できません。")    print(f"       period を短くしてください（現在: {period}）。データ取得に失敗します。")else:    print(f"設定OK: {ticker} / {period} / {interval}足")    print(f"  レンジ{or_minutes}分, 損切り{stop_mult}倍, 利確{target_mult}倍, "          f"ショート{'あり' if allow_short else 'なし'}")    print(f"  資金{capital:,}円, リスク{risk_pct}%/回")

## 3. バックテスト実行

Yahoo Finance から実データを取得して検証します。
`--fallback error` を付けているので、**データが取れなかった場合は
ダミーデータで代用せずエラー終了**します。


In [ ]:
import subprocess, sysargs = [    sys.executable, "ntt_daytrade_backtest.py",    "--ticker", ticker,    "--period", period,    "--interval", interval,    "--or-minutes", str(or_minutes),    "--stop-mult", str(stop_mult),    "--target-mult", str(target_mult),    "--buffer-bps", str(buffer_bps),    "--capital", str(capital),    "--risk-pct", str(risk_pct / 100.0),    "--commission-bps", str(commission_bps),    "--slippage-bps", str(slippage_bps),    "--fallback", "error",    "--save-data", "ntt_bars.csv",]if not allow_short:    args.append("--no-short")proc = subprocess.run(args, capture_output=True, text=True)print(proc.stdout)if proc.returncode != 0:    print("--- エラー出力 ---")    print(proc.stderr[-2000:])    print("\n取得に失敗しました。時間をおくか、period / interval を変えて再実行してください。")

## 4. 資産曲線とドローダウン


In [ ]:
import osfrom IPython.display import Image, displaypng = "ntt_daytrade_equity.png"if os.path.exists(png):    display(Image(png))else:    print("グラフがまだありません。3. のセルを先に実行してください。")

## 5. トレード履歴


In [ ]:
import osimport pandas as pdpd.set_option("display.max_rows", 200)if os.path.exists("ntt_daytrade_trades.csv"):    trades = pd.read_csv("ntt_daytrade_trades.csv")    print(f"トレード数: {len(trades)}")    display(trades.tail(30))else:    print("トレード履歴がまだありません。3. のセルを先に実行してください。")

## 6. パラメータ感度チェック（任意）

レンジ幅・損切り・利確の組み合わせを総当たりします。
**イン・サンプルの結果なので、ここで最も良い数値を選ぶのは過剰最適化です。**
戦略が特定の設定にどれだけ依存しているかを見るために使ってください。


In [ ]:
import os, subprocess, sysif not os.path.exists("ntt_bars.csv"):    raise SystemExit("先に 3. のセルを実行して ntt_bars.csv を作成してください。")# 保存済みのバーを使うので再ダウンロードは不要sweep = subprocess.run(    [sys.executable, "ntt_daytrade_backtest.py",     "--csv", "ntt_bars.csv", "--sweep",     "--capital", str(capital), "--risk-pct", str(risk_pct / 100.0),     "--out-prefix", "sweep"],    capture_output=True, text=True,)out = sweep.stdoutprint(out[out.find("Parameter sweep"):] if "Parameter sweep" in out else out or sweep.stderr[-1500:])

## 7. 結果をiPadに保存（任意）


In [ ]:
from google.colab import filesimport osfor f in ["ntt_daytrade_trades.csv", "ntt_daytrade_equity.png", "ntt_bars.csv"]:    if os.path.exists(f):        files.download(f)

---

## 補足

* Yahoo Finance の分足は履歴が短く、**5分足で約60日、1分足で約7日**が上限です。
  長期の検証はできません。
* 60営業日という標本数はデイトレ戦略の優位性を判断するには少なめです。
  結果は幅を持って見てください。
* 約定条件は保守側に倒しています（スリッページは常に不利方向、
  同じ足で損切りと利確の両方に触れた場合は損切り優先）。
* `ntt_bars.csv` を保存しておけば、`--csv ntt_bars.csv` でオフライン再実行できます。
